In [ ]:
from functools import partial
import jax
import jax.numpy as jnp
from flax import nnx
from einops import rearrange, reduce, repeat
import optax
from optax import softmax_cross_entropy_with_integer_labels as softmax_ce
import mnist1d

In [ ]:
def sigreg_loss(embs, num_slices, rngs):
    bs, v, d = embs.shape

    A = jax.random.normal(rngs.next(), (d, num_slices))
    A = A / jnp.linalg.norm(A, axis = 0, keepdims = True)

    t = jnp.linspace(-3, 3, 17)

    # theoretical gaussian CF and w(t) weighting
    exp_f = jnp.exp(-0.5 * t**2)

    projs = embs @ A

    # Using exp(ix) = cos(x) + i sin(x) and 
    x_t = rearrange(projs, 'b v m -> b v m 1') * t
    x_t_cos = reduce(jnp.cos(x_t), 'b v m t -> b m t', 'mean') # important: mean over views
    x_t_sin = reduce(jnp.sin(x_t), 'b v m t -> b m t', 'mean')

    err = jnp.square(x_t_cos - exp_f) + jnp.square(x_t_sin - 0)

    EP = v * jnp.trapezoid(err * exp_f, t, axis = -1)
    return reduce(EP, 'b m -> ', 'mean')


def loss_fn(encoder, vs, num_slices, lamb, rngs):

    embs = encoder(vs) # (bs, v, d)

    centers = reduce(embs, 'b v d -> b 1 d', 'mean')
    pred_loss = jnp.square(embs - centers).mean()


    reg_loss = sigreg_loss(embs, num_slices = num_slices, rngs = rngs)
    loss = lamb * reg_loss + (1-lamb) * pred_loss
    return loss, (embs, pred_loss, reg_loss)


grad_fn = nnx.value_and_grad(loss_fn, has_aux=True)


def probe_loss(probe, embs, y):
    _, v, *_ = embs.shape
    return softmax_ce(
        logits = rearrange(probe(embs), 'b v d -> (b v) d'),
        labels = repeat(y, 'b -> (b v)', v = v)
    ).mean()

probe_grad_fn = nnx.value_and_grad(probe_loss)

In [ ]:
ds = mnist1d.data.make_dataset()
l = 40 # 1d input length


# def gen_views(x, rngs):
#     bs, l = x.shape

#     # small per-example shifts
#     w = 8
#     shifts = jax.random.randint(rngs.next(), (bs,), -w, w + 1)
#     v2 = jax.vmap(lambda xi, s: jnp.roll(xi, s))(x, shifts)

#     # optional small amplitude jitter
#     scale = 1.0 + 0.3 * jax.random.normal(rngs.next(), (bs, 1))
#     v2 = v2 * scale

#     # optional small iid noise
#     v2 = v2 + 0.02 * jax.random.normal(rngs.next(), v2.shape)

#     return jnp.stack([x, v2], axis=1)

def gen_views(x, rngs, V=4):
    """
    Mirrors mnist1d's generative transforms: shift, scale (amplitude),
    correlated (low-freq) noise, iid noise, and random masking.
    x: (bs, l) -> (bs, V, l)
    """
    bs, l = x.shape

    def one_view(key):
        k_shift, k_scale, k_corr, k_iid, k_mask_pos, k_mask_len, k_flip = jax.random.split(key, 7)

        # 1. Shift (mnist1d default max_translation is ~8 on length 40)
        shifts = jax.random.randint(k_shift, (bs,), -8, 9)
        v = jax.vmap(lambda xi, s: jnp.roll(xi, s))(x, shifts)

        # 2. Amplitude scale (per-example)
        scale = 1.0 + 0.3 * jax.random.normal(k_scale, (bs, 1))
        v = v * scale

        # 3. Correlated (low-frequency) noise: gaussian-smoothed noise
        #    approximates mnist1d's corr_noise by blurring white noise
        raw = jax.random.normal(k_corr, (bs, l))
        # simple box-blur via convolution (kernel size 7)
        kernel = jnp.ones((7,)) / 7
        corr = jax.vmap(lambda r: jnp.convolve(r, kernel, mode='same'))(raw)
        v = v + 0.25 * corr

        # 4. iid noise
        v = v + 0.05 * jax.random.normal(k_iid, (bs, l))

        # 5. Random masking: zero out a contiguous chunk (length-invariance)
        mask_len = jax.random.randint(k_mask_len, (bs,), 0, 8)   # 0-7 zeros
        mask_pos = jax.random.randint(k_mask_pos, (bs,), 0, l)
        idx = jnp.arange(l)[None, :]                             # (1, l)
        mask = (idx >= mask_pos[:, None]) & (idx < (mask_pos + mask_len)[:, None])
        v = jnp.where(mask, 0.0, v)

        return v

    keys = jax.random.split(rngs.next(), V)
    views = jnp.stack([one_view(k) for k in keys], axis=1)  # (bs, V, l)
    return views

def train_loader(bs, epochs = 100):
    for _ in range(epochs):
        for i in range(len(ds['x']) // bs):
            yield ds['x'][i*bs:(i+1)*bs], ds['y'][i*bs:(i+1)*bs]

def test_loader(bs):
    for i in range(len(ds['x_test']) // bs):
        yield ds['x_test'][i*bs:(i+1)*bs], ds['y_test'][i*bs:(i+1)*bs]

def test_acc(model, loader, bs):
    correct, total = 0, 0
    for x, y in loader(bs):
        logits = model(x)
        pred = jnp.argmax(logits, axis=-1)
        correct += (pred == y).sum()
        total += len(y)
    return correct / total

In [ ]:
h_dim, emb_dim = 64, 16
num_slices = 128
lamb = 0.1
bs = 32
seed = 0
rngs = nnx.Rngs(seed)

# enc = nnx.Sequential(
#     partial(rearrange, pattern = 'b ... l -> b ... l 1'),
#     nnx.Conv(1, 32, kernel_size=(5,), padding='SAME', rngs=rngs),
#     nnx.relu,
#     nnx.Conv(32, emb_dim, kernel_size=(5,), padding='SAME', rngs=rngs),
#     partial(reduce, pattern = 'b ... l d -> b ... d', reduction = 'mean'), # pool over length
#     nnx.BatchNorm(emb_dim, rngs=rngs, use_bias=False, use_scale=False)
# )
enc = nnx.Sequential(
    partial(rearrange, pattern='b ... l -> b ... l 1'),
    nnx.Conv(1, 64, kernel_size=(5,), padding='SAME', rngs=rngs),
    nnx.relu,
    nnx.Conv(64, 64, kernel_size=(5,), strides=(2,), padding='SAME', rngs=rngs),  # l=20
    nnx.relu,
    nnx.Conv(64, 128, kernel_size=(5,), strides=(2,), padding='SAME', rngs=rngs), # l=10
    nnx.relu,
    nnx.Conv(128, 128, kernel_size=(5,), padding='SAME', rngs=rngs),
    partial(reduce, pattern='b ... l d -> b ... d', reduction='mean'),
    nnx.BatchNorm(128, rngs=rngs, use_bias=False, use_scale=False)
    # representation dim = 128
)
emb_dim = 128
probe = nnx.Linear(emb_dim, 10, rngs = rngs)

# baseline: cross-entropy classifier without stop gradient
clf_enc, clf_probe = nnx.clone(enc), nnx.clone(probe)
clf = nnx.Sequential(clf_enc, clf_probe)


opt_enc = nnx.Optimizer(enc, optax.adam(3e-4), wrt = nnx.Param)
opt_probe = nnx.Optimizer(probe, optax.adam(3e-4), wrt = nnx.Param)
opt_clf = nnx.Optimizer(clf, optax.adam(3e-3), wrt = nnx.Param)

def clf_loss(clf, x, y):
    return softmax_ce(clf(x), y).mean()
clf_grad_fn = nnx.value_and_grad(clf_loss)


for step, (x, y) in enumerate(train_loader(bs)):

    vs = gen_views(x, rngs)

    (lejepa_loss, (embs, pred_loss, reg_loss)), grads = grad_fn(enc, vs, num_slices = num_slices, lamb = lamb, rngs = rngs)
    opt_enc.update(enc, grads)

    # probe is just a diagnostic, so we don't backprop through the encoder
    embs = jax.lax.stop_gradient(embs)
    probe_loss_val, probe_grads = probe_grad_fn(probe, embs, y)
    opt_probe.update(probe, probe_grads)

    clf_loss_val, clf_grads = clf_grad_fn(clf, x, y)
    opt_clf.update(clf, clf_grads)

    # print(lejepa_loss, pred_loss, reg_loss, probe_loss_val, clf_loss_val, jnp.std(rearrange(embs, 'b v d -> (b v) d'), axis=0).mean())
    
    if step % 50 == 0:
        flat = rearrange(embs, 'b v d -> (b v) d')
        per_dim_std = jnp.std(flat, axis=0)
        print(f"step={step} pred={pred_loss:.4f} reg={reg_loss:.4f} "
            f"probe={probe_loss_val:.4f} clf={clf_loss_val:.4f} "
            f"std_min={per_dim_std.min():.3f} std_mean={per_dim_std.mean():.3f}")

In [ ]:
test_acc(clf, test_loader, bs)

In [ ]:
jnp.std(embs, axis = 1).mean()

In [ ]:
test_acc(probe, test_loader, bs)

In [ ]:
enc(vs).shape, clf(vs).shape

In [ ]:
plt.plot(v1[0])
plt.plot(v2[0])